In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4), layout="constrained")

t_grid = np.linspace(-0.5, 1.5, 1024)
ax.plot(t_grid, np.sin(2 * np.pi * t_grid), marker="")
ax.plot(t_grid, np.sin(2 * 2 * np.pi * t_grid), marker="")
ax.plot(t_grid, np.sin(4 * 2 * np.pi * t_grid), marker="")
ax.set_frame_on(False)
ax.set_xticks([])
ax.set_yticks([])

fig.savefig("sinusoids.pdf")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6), layout="constrained")

ellipse = mpl.patches.Ellipse(
    xy=(0, 0), width=4, height=2, angle=45, edgecolor="r", fc="None", lw=2
)

ax.add_patch(ellipse)

ax.set_frame_on(False)
ax.set_xticks([])
ax.set_yticks([])

ax.set(xlim=(-5, 5), ylim=(-5, 5))

# fig.savefig("ellipses.pdf")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from numpy.linalg import solve

# Three data points (chosen to look nice)
pts = np.array([[1.0, 0.0], [0, 0.5], [0.0, -0.5]])

# General conic: Ax^2 + Bxy + Cy^2 + Dx + Ey + F = 0
# Normalize F = 1. Each point gives one equation:
#   x^2 A + xy B + y^2 C + x D + y E = -1
# That's 3 equations in 5 unknowns → 2 free params.
# Build the 3x5 matrix:
M = np.array([[x**2, x * y, y**2, x, y] for x, y in pts])

# Particular solution (set B=0, D=0, solve for A, C, E):
# M[:, [0,2,4]] @ [A, C, E] = -1
M_sub = M[:, [0, 2, 4]]
particular = solve(M_sub, -np.ones(3))  # [A, C, E]


# Null space basis: two vectors spanning the free params B, D
# Full solution: [A,B,C,D,E] = particular([0,2,4]) + s*n1 + t*n2
def make_conic(s, t):
    """Return (A,B,C,D,E) for free parameters s, t."""
    # particular fills slots 0,2,4; null directions fill slots 1,3
    n1 = np.zeros(5)
    n1_rhs = -M[:, 1]  # coefficient of B
    n1[[0, 2, 4]] = solve(M_sub, n1_rhs)
    n1[1] = 1.0

    n2 = np.zeros(5)
    n2_rhs = -M[:, 3]  # coefficient of D
    n2[[0, 2, 4]] = solve(M_sub, n2_rhs)
    n2[3] = 1.0

    p = np.zeros(5)
    p[[0, 2, 4]] = particular
    return p + s * n1 + t * n2


def plot_conic(ax, coeffs, color, npts=500):
    A, B, C, D, E = coeffs
    # Check ellipse condition
    if B**2 - 4 * A * C >= 0:
        return False
    # Parameterize by angle, solve for r
    theta = np.linspace(0, 2 * np.pi, npts)
    curves = []
    for th in theta:
        ct, st = np.cos(th), np.sin(th)
        # A(r ct)^2 + B(r ct)(r st) + C(r st)^2 + D(r ct) + E(r st) + 1 = 0
        a_coef = A * ct**2 + B * ct * st + C * st**2
        b_coef = D * ct + E * st
        c_coef = 1.0
        disc = b_coef**2 - 4 * a_coef * c_coef
        if disc < 0:
            curves.append((np.nan, np.nan))
            continue
        r = (-b_coef + np.sqrt(disc)) / (2 * a_coef)
        if r > 0:
            curves.append((r * ct, r * st))
        else:
            r = (-b_coef - np.sqrt(disc)) / (2 * a_coef)
            curves.append((r * ct, r * st) if r > 0 else (np.nan, np.nan))
    xs, ys = zip(*curves)
    ax.plot(xs, ys, color=color, marker="")
    return True


# Pick three (s, t) values that give qualitatively different ellipses
# (you'll need to fiddle with these to taste)
params = [(0.0, 0), (2.0, 5.0), (-2, 5), (0, 3)]  # , (1, 1)]
colors = ["C0", "C1", "C2", "C3"]

fig, ax = plt.subplots(figsize=(6, 6), layout="constrained")
for (s, t), c in zip(params, colors):
    coeffs = make_conic(s, t)
    plot_conic(ax, coeffs, c)

ax.plot(*pts.T, "o", color="w", ms=8, zorder=5)
ax.set_aspect("equal")
ax.set_frame_on(False)
ax.set_xticks([])
ax.set_yticks([])

fig.savefig("ellipses.pdf")